# Stacking Under Distribution Shift
## ASTraM Traffic Incident Prediction
This notebook evaluates whether model stacking (and specifically TabPFN's contribution) survives strict distribution shifts via Temporal CV and GroupKFold (by corridor).

**Note:** This notebook relies entirely on pre-computed Out-of-Fold (OOF) predictions to guarantee honesty. No synthetic fallback data is used.

In [1]:
import os
import sys
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import StratifiedKFold, TimeSeriesSplit, GroupKFold
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.preprocessing import StandardScaler
from lightgbm import LGBMClassifier
import warnings
warnings.filterwarnings('ignore')

SEED = 0

# ---------------------------------------------------------
# 1. VALIDATE AND LOAD REQUIRED OOF FILES
# ---------------------------------------------------------
schemes = ['random', 'temporal', 'corridor']
base_models = ['CatBoost', 'LightGBM', 'XGBoost', 'RandomForest', 'ExtraTrees', 'Logistic']

required_files = ['Astram event data_anonymized - Astram event data_anonymizedb40ac87.csv']
for s in schemes:
    for m in base_models:
        required_files.append(f"oof_{s}_{m}.npy")
    required_files.append(f"oof_tabpfn_{s}.npy")

missing_files = [f for f in required_files if not os.path.exists(f)]

if missing_files:
    raise FileNotFoundError(f"Missing {len(missing_files)} required OOF files. Please ensure the following files exist in the directory before running:\n" + "\n".join(missing_files))

print("All required OOF files found.")


All required OOF files found.


In [2]:
# ---------------------------------------------------------
# 2. LOAD GROUND TRUTH (Aligned identically to OOF generation)
# ---------------------------------------------------------
DATA = 'Astram event data_anonymized - Astram event data_anonymizedb40ac87.csv'
df = pd.read_csv(DATA, low_memory=False)
df['t'] = pd.to_datetime(df['created_date'], errors='coerce')
df = df.sort_values('t').reset_index(drop=True)

y = df['requires_road_closure'].astype(int).values
corridor = df['corridor'].astype(str).fillna('NA').values
N = len(df)

print(f"Loaded ground truth. Total rows: {N}")


Loaded ground truth. Total rows: 8173


In [3]:
# ---------------------------------------------------------
# 3. EVALUATION AND STACKING LOGIC
# ---------------------------------------------------------
META = ['Logistic', 'Ridge', 'LightGBM']
BASE_ORDER = base_models + ['TabPFN']

def get_meta_folds(scheme, idx):
    if scheme == 'random':
        return list(StratifiedKFold(5, shuffle=True, random_state=SEED).split(idx, y[idx]))
    if scheme == 'corridor':
        return list(GroupKFold(5).split(idx, y[idx], groups=corridor[idx]))
    return list(TimeSeriesSplit(5).split(idx))

def fit_meta(kind, Xtr, ytr, Xte):
    if kind == 'Logistic':
        sc = StandardScaler().fit(Xtr)
        m = LogisticRegression(max_iter=1000, class_weight='balanced').fit(sc.transform(Xtr), ytr)
        return m.predict_proba(sc.transform(Xte))[:, 1]
    if kind == 'Ridge':
        sc = StandardScaler().fit(Xtr)
        m = Ridge(alpha=1.0).fit(sc.transform(Xtr), ytr)
        return m.predict(sc.transform(Xte))
    if kind == 'LightGBM':
        m = LGBMClassifier(n_estimators=200, num_leaves=15, learning_rate=0.05,
                           random_state=SEED, n_jobs=-1, verbose=-1).fit(Xtr, ytr)
        return m.predict_proba(Xte)[:, 1]

def evaluate_scheme(scheme):
    # Load OOFs
    OOF = {}
    for m in base_models:
        OOF[m] = np.load(f"oof_{scheme}_{m}.npy")
    OOF['TabPFN'] = np.load(f"oof_tabpfn_{scheme}.npy")
    
    # Calculate coverage mask (to ignore NaNs if any)
    base_cov = ~np.isnan(np.column_stack([OOF[m] for m in BASE_ORDER])).any(axis=1)
    idx = np.where(base_cov)[0]
    
    # Base standalone performance on this mask
    yv = y[idx]
    standalone = {}
    for m in BASE_ORDER:
        p = OOF[m][idx]
        standalone[m] = {
            'auc': roc_auc_score(yv, p),
            'ap': average_precision_score(yv, p)
        }
        
    def evaluate_stack(cols):
        B = np.column_stack([OOF[m][idx] for m in cols])
        out = {}
        for kind in META:
            pred = np.full(len(idx), np.nan)
            per = []
            for tr, te in get_meta_folds(scheme, idx):
                p = fit_meta(kind, B[tr], yv[tr], B[te])
                pred[te] = p
                per.append((roc_auc_score(yv[te], p), average_precision_score(yv[te], p)))
            cov = ~np.isnan(pred)
            per = np.array(per)
            out[kind] = {
                'auc': roc_auc_score(yv[cov], pred[cov]),
                'ap': average_precision_score(yv[cov], pred[cov]),
                'auc_sd': per[:,0].std(),
                'ap_sd': per[:,1].std()
            }
        return out
        
    stack_full = evaluate_stack(BASE_ORDER)
    stack_notab = evaluate_stack([m for m in BASE_ORDER if m != 'TabPFN'])
    
    return standalone, stack_full, stack_notab, len(idx)


In [4]:
# ---------------------------------------------------------
# 4. RUN EXPERIMENTS
# ---------------------------------------------------------
results = {}
for s in schemes:
    print(f"Evaluating {s} scheme...")
    results[s] = evaluate_scheme(s)


Evaluating random scheme...
Evaluating temporal scheme...
Evaluating corridor scheme...


In [5]:
# ---------------------------------------------------------
# 5. REPORTING TABLES
# ---------------------------------------------------------
import pandas as pd
from IPython.display import display

# TABLE A: Best standalone model under each CV scheme
table_a_data = []
for s in schemes:
    standalone = results[s][0]
    best_m = max(standalone.keys(), key=lambda k: standalone[k]['ap'])
    table_a_data.append({
        'CV': s,
        'Model': best_m,
        'ROC-AUC': standalone[best_m]['auc'],
        'PR-AUC': standalone[best_m]['ap']
    })
table_a = pd.DataFrame(table_a_data)
print("\nTABLE A: Best Standalone Model Under Each CV Scheme")
display(table_a)

# TABLE B: Stack performance under each CV scheme (Best Stack)
table_b_data = []
for s in schemes:
    stack_full = results[s][1]
    for meta in META:
        table_b_data.append({
            'CV': s,
            'Meta_Learner': meta,
            'ROC-AUC': stack_full[meta]['auc'],
            'PR-AUC': stack_full[meta]['ap'],
            'AUC_std': stack_full[meta]['auc_sd'],
            'AP_std': stack_full[meta]['ap_sd']
        })
table_b = pd.DataFrame(table_b_data)
print("\nTABLE B: Stack Performance Under Each CV Scheme (With TabPFN)")
display(table_b)

# TABLE C: Lift of stack over best standalone
table_c_data = []
for s in schemes:
    standalone = results[s][0]
    best_standalone_m = max(standalone.keys(), key=lambda k: standalone[k]['ap'])
    best_standalone_ap = standalone[best_standalone_m]['ap']
    
    stack_full = results[s][1]
    best_meta_m = max(META, key=lambda k: stack_full[k]['ap'])
    best_stack_ap = stack_full[best_meta_m]['ap']
    
    table_c_data.append({
        'CV': s,
        'Best_Standalone_PR': best_standalone_ap,
        'Best_Stack_PR': best_stack_ap,
        'Lift_PR': best_stack_ap - best_standalone_ap
    })
table_c = pd.DataFrame(table_c_data)
print("\nTABLE C: Lift of Stack Over Best Standalone (Using best meta)")
display(table_c)

# TABLE D: TabPFN contribution under each CV scheme
table_d_data = []
for s in schemes:
    stack_full = results[s][1]
    stack_notab = results[s][2]
    
    # Compare using the best meta-learner for the full stack
    best_meta_m = max(META, key=lambda k: stack_full[k]['ap'])
    
    full_ap = stack_full[best_meta_m]['ap']
    notab_ap = stack_notab[best_meta_m]['ap']
    
    table_d_data.append({
        'CV': s,
        'Meta_Learner': best_meta_m,
        'Stack_No_TabPFN_PR': notab_ap,
        'Stack_With_TabPFN_PR': full_ap,
        'TabPFN_Contribution_PR': full_ap - notab_ap
    })
table_d = pd.DataFrame(table_d_data)
print("\nTABLE D: TabPFN Contribution Under Each CV Scheme")
display(table_d)



TABLE A: Best Standalone Model Under Each CV Scheme


,CV,Model,ROC-AUC,PR-AUC
0,random,RandomForest,0.808406,0.423017
1,temporal,CatBoost,0.756168,0.318228
2,corridor,XGBoost,0.711877,0.291667



TABLE B: Stack Performance Under Each CV Scheme (With TabPFN)


,CV,Meta_Learner,ROC-AUC,PR-AUC,AUC_std,AP_std
0,random,Logistic,0.814847,0.442180,0.012099,0.057409
1,random,Ridge,0.818810,0.444984,0.014130,0.060775
2,random,LightGBM,0.808042,0.411744,0.010686,0.039964
3,temporal,Logistic,0.770645,0.312557,0.048114,0.069746
4,temporal,Ridge,0.772149,0.321546,0.048195,0.069804
5,temporal,LightGBM,0.743484,0.262263,0.048758,0.053862
6,corridor,Logistic,0.738433,0.302557,0.022872,0.093865
7,corridor,Ridge,0.715115,0.266661,0.029245,0.087647
8,corridor,LightGBM,0.690362,0.270064,0.010938,0.062617



TABLE C: Lift of Stack Over Best Standalone (Using best meta)


,CV,Best_Standalone_PR,Best_Stack_PR,Lift_PR
0,random,0.423017,0.444984,0.021967
1,temporal,0.318228,0.321546,0.003317
2,corridor,0.291667,0.302557,0.010890



TABLE D: TabPFN Contribution Under Each CV Scheme


,CV,Meta_Learner,Stack_No_TabPFN_PR,Stack_With_TabPFN_PR,TabPFN_Contribution_PR
0,random,Ridge,0.438086,0.444984,0.006898
1,temporal,Ridge,0.327750,0.321546,-0.006204
2,corridor,Logistic,0.320976,0.302557,-0.018420


In [6]:
# ---------------------------------------------------------
# 6. FINAL VERDICT
# ---------------------------------------------------------
survives = all(row['Lift_PR'] > 0 for _, row in table_c.iterrows() if row['CV'] in ['temporal', 'corridor'])

print('='*78)
print('VERDICT')
print('='*78)

for _, row in table_c.iterrows():
    tag = 'beats' if row['Lift_PR'] > 0 else 'does NOT beat'
    print(f"  {row['CV']:8s}: stack PR-AUC={row['Best_Stack_PR']:.4f} vs standalone {row['Best_Standalone_PR']:.4f}  -> lift {row['Lift_PR']:+.4f} [{tag}]")

print()
if survives:
    print('  >>> (A) STACK SURVIVES DISTRIBUTION SHIFT')
    print('      Stack PR-AUC exceeds best standalone under BOTH temporal and corridor CV.')
else:
    print('  >>> (B) STACK ONLY WORKS UNDER RANDOM CV')
    print('      Under at least one shifted scheme the stack does not beat the best single model.')


VERDICT
  random  : stack PR-AUC=0.4450 vs standalone 0.4230  -> lift +0.0220 [beats]
  temporal: stack PR-AUC=0.3215 vs standalone 0.3182  -> lift +0.0033 [beats]
  corridor: stack PR-AUC=0.3026 vs standalone 0.2917  -> lift +0.0109 [beats]

  >>> (A) STACK SURVIVES DISTRIBUTION SHIFT
      Stack PR-AUC exceeds best standalone under BOTH temporal and corridor CV.
